# Stage 1: data

**Goal.** Turn a dataset's raw files into the two formats every later stage reads: a *patch cache*
for training and whole *test scenes* for evaluation. After this stage nothing depends on which
dataset is in use, which is how "only the dataset changes" is kept true.

**Command-line equivalent:** `p037 data --config <config> [--source <folder or archive>]`

**Code:** `src/p037/datasets/` (`base.py` for the formats, one module per dataset)

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from p037 import plots                      # sets the figure style used in the report
from p037.config import REPO_ROOT, load_config
from p037.datasets import get_dataset

# The experiment this notebook works on. smoke.yaml is generated data: it runs anywhere in minutes
# and its numbers mean nothing. On BluePebble use analysis/p037/configs/exp_b_usf.yaml.
# (Setting the environment variable P037_CONFIG before starting Jupyter does the same for all notebooks.)
CONFIG = os.environ.get("P037_CONFIG", "analysis/p037/configs/smoke.yaml")
cfg = load_config(REPO_ROOT / CONFIG)
ds = get_dataset(cfg)
DRY_RUN = cfg.dataset == "synthetic"
print(f"experiment {cfg.name!r} on dataset {cfg.dataset!r}")
print("dataset files:", cfg.dataset_dir)
print("run folder:   ", cfg.out)

## 1.1 Build the patch cache

For UrbanSARFloods this reads 6,471 chips of 512 px, converts the four intensity bands to the
model's 8-bit scale, cuts each chip into four 256 px patches and writes them to two large arrays.
It takes a while and 9 GB, so on BluePebble run it as a job (`sbatch data.sbatch <config>`) and
use this notebook to inspect the result. The cell below builds the cache here only for the dry
run, or if you point `P037_SOURCE` at the dataset on disk.

In [ ]:
SOURCE = os.environ.get("P037_SOURCE")       # extracted dataset folder or the .tar.gz archive

if ds.has_cache():
    print("cache already built:", ds.cache_dir)
elif DRY_RUN or SOURCE:
    ds.build_cache(SOURCE)
else:
    print("No cache yet. Build it with the data job, or set P037_SOURCE and run this cell again.")

## 1.2 What is in the cache

`meta.json` names the stored channels. Configs ask for channels **by name**, never by position:
the UrbanSARFloods files carry no band names and do not follow the paper's order
(`analysis/band_identification.md`), so positions are exactly where a silent mistake would hide.

In [ ]:
meta = ds.meta()
print("channels, in stored order:", meta.channels)
print("classes:", meta.classes, "| patch size:", meta.patch, "px")
print("this experiment feeds the model:", cfg.input_channels)

patches = ds.patches()
patches.head()

In [ ]:
# patches per event and split
patches.pivot_table(index="event", columns="split", values="patch_id", aggfunc="count", fill_value=0)

## 1.3 Class balance

Flood is rare. This table is the reason the training stage keeps the checkpoint with the best
validation **F1** and not the best accuracy: a model that never predicts flood is already almost
always "right".

In [ ]:
from p037.datasets.urbansarfloods import class_balance

class_balance(patches, meta.classes)

## 1.4 One patch, as the model sees it

The cache stores backscatter as 8-bit numbers: `(max(-49, min(1, dB)) + 50) * 5`. That is the
scaling the U-Net was designed for. `uint8_to_db` turns it back into decibels for plotting.

Open water is a smooth surface that reflects the radar pulse away from the sensor, so it is dark.
A flood therefore shows as a patch that is dark **after** the event and was not dark **before**.

In [ ]:
from p037.datasets.base import INVALID, uint8_to_db

images, labels = ds.open_arrays("train")     # memory-mapped: nothing is loaded until it is used
train_rows = patches[patches["split"] == "train"]
flood_px = sum(train_rows[f"n_{c}"] for c in meta.classes.values())
row = train_rows.loc[flood_px.idxmax()]      # the training patch with the most flood
img, lab = np.asarray(images[row["index"]]), np.asarray(labels[row["index"]])

fig, axes = plt.subplots(1, len(meta.channels) + 1, figsize=(3 * (len(meta.channels) + 1), 3.3))
for ax, name, band in zip(axes, meta.channels, uint8_to_db(img)):
    im = ax.imshow(band, cmap="gray", vmin=-25, vmax=0)
    ax.set_title(f"{name} (dB)", loc="left")
shown = np.ma.masked_equal(lab, INVALID)
axes[-1].imshow(shown, cmap=plots.SEQ, vmin=0, vmax=max(meta.classes))
axes[-1].set_title("label " + str(meta.classes), loc="left")
for ax in axes:
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(visible=False)
fig.suptitle(row["patch_id"], x=0.01, ha="left")
fig.tight_layout()

## 1.5 Does the signal exist? Backscatter of flooded and dry pixels

A quick physical check on a sample of patches: after the event, flooded pixels should sit lower
(darker) than the rest; before the event they should not. If this plot ever shows the opposite,
the band order is wrong and nothing downstream can be trusted.

In [ ]:
rng = np.random.default_rng(0)
with_flood = train_rows[flood_px > 0]
sample = with_flood.sample(min(150, len(with_flood)), random_state=0)["index"].to_numpy()
img_s = uint8_to_db(np.stack([np.asarray(images[i]) for i in np.sort(sample)]))    # (n, C, P, P)
lab_s = np.stack([np.asarray(labels[i]) for i in np.sort(sample)])
flooded, dry = np.isin(lab_s, list(meta.classes)), lab_s == 0

fig, axes = plt.subplots(1, len(meta.channels), figsize=(3 * len(meta.channels), 2.8), sharey=True)
bins = np.linspace(-35, 5, 60)
for ax, c, name in zip(axes, range(len(meta.channels)), meta.channels):
    ax.hist(img_s[:, c][dry], bins=bins, density=True, color=plots.MUTED, alpha=0.6, label="not flooded")
    ax.hist(img_s[:, c][flooded], bins=bins, density=True, color=plots.BLUE, alpha=0.6, label="flooded")
    ax.set_title(name, loc="left"); ax.set_xlabel("backscatter (dB)")
axes[0].set_ylabel("density"); axes[0].legend()
fig.tight_layout()
pd.DataFrame({"flooded median dB": [np.nanmedian(img_s[:, c][flooded]) for c in range(len(meta.channels))],
              "not flooded median dB": [np.nanmedian(img_s[:, c][dry]) for c in range(len(meta.channels))]},
             index=meta.channels).round(1)

## 1.6 Invalid pixels and the split used for training

Pixels without SAR data (swath edges) carry label 255. They are left out of the loss and of every
score. Patches without a single valid pixel are dropped; partly valid ones are kept.

`select_patches` applies the experiment's validation choice (`val_mode` in the config):
`official` uses the dataset's own lists, `events` holds out whole events.

In [ ]:
from p037.data import select_patches

total_px = len(patches) * meta.patch ** 2
print(f"invalid pixels: {100 * (1 - patches['n_valid'].sum() / total_px):.1f} % of all cached pixels")
print(f"patches without any valid pixel: {(patches['n_valid'] == 0).sum()} of {len(patches)}")
train_sel, val_sel = select_patches(cfg, ds)
print(f"validation mode {cfg.val_mode!r}: {len(train_sel)} training patches, {len(val_sel)} validation patches")
print("validation events:", sorted(val_sel["event"].unique())[:8], "...")

## 1.7 Test scenes

The held-out scenes are kept whole, each with its label and a valid-data mask on the same grid.
Evaluation predicts them in large windows and scores 256 px tiles.

For UrbanSARFloods the four scenes are separate downloads (31 GB), so again: use the data job on
BluePebble, and this cell to look at the result.

In [ ]:
import rasterio

SCENES = os.environ.get("P037_SCENES")       # folder holding the raw test scenes, if on disk
events = cfg.eval_events or ds.list_events(SCENES)
for event in events:
    if not ds.has_scene(event) and (DRY_RUN or SCENES):
        ds.prepare_scene(event, SCENES)
ready = [e for e in events if ds.has_scene(e)]
print("prepared scenes:", ready, "| still to prepare:", [e for e in events if e not in ready])

if ready:
    fig, axes = plt.subplots(len(ready), 2, figsize=(9, 3.4 * len(ready)), squeeze=False)
    for (a_img, a_lab), event in zip(axes, ready):
        f = ds.scene_files(event)
        with rasterio.open(f["image"]) as im, rasterio.open(f["gt"]) as gt:
            step = max(1, max(im.height, im.width) // 800)         # read a reduced overview, not the full scene
            shape = (im.height // step, im.width // step)
            last = im.count                                         # the last stored channel
            a_img.imshow(uint8_to_db(im.read(last, out_shape=shape)), cmap="gray", vmin=-25, vmax=0)
            a_img.set_title(f"{event}: {im.descriptions[last - 1]} (dB), {im.height} x {im.width} px", loc="left")
            g = gt.read(1, out_shape=shape)
            a_lab.imshow(np.ma.masked_equal(g, INVALID), cmap=plots.SEQ, vmin=0, vmax=max(meta.classes))
            a_lab.set_title("label (blank = invalid)", loc="left")
        for ax in (a_img, a_lab):
            ax.set_xticks([]); ax.set_yticks([]); ax.grid(visible=False)
    fig.tight_layout()

## Check before moving on

- The channel names and the class balance match what you expect for the dataset.
- Flooded pixels are darker after the event (section 1.5).
- Every test scene is prepared.

Next: `02_train.ipynb`.